In [ ]:
# ============================================================
# 03_Model_Training.ipynb
# SIMPLE APPLE-LEVEL SPECTRAL TRAINING
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from scipy.io import loadmat

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print("=" * 80)
print("SpectroFood Apple Maturity - SIMPLE Spectral-Only Training (Apple-level)")
print("=" * 80)
print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# ---------------- CONFIG ----------------
class Config:
    APPLE_MAT_PATH = "/content/Apple.mat"               # A — already uploaded
    CSV_PATH = "/content/SpectroFood_dataset.csv"       # A — already uploaded

    NUM_APPLES = 240
    NUM_BANDS = 141
    NUM_CLASSES = 3

    TEST_SIZE = 0.2
    RANDOM_STATE = 42

    BATCH_SIZE = 16
    EPOCHS = 200
    LEARNING_RATE = 1e-3

    CHECKPOINT_DIR = "checkpoints_simple"
    RESULTS_DIR = "results_simple"

config = Config()
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)


SpectroFood Apple Maturity - Hybrid CNN+Transformer Training
TensorFlow Version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ============================================================
# OPTIONAL: GCS DOWNLOAD (YOU DO NOT NEED THIS NOW)
# ============================================================

"""
from google.colab import auth
from google.cloud import storage

auth.authenticate_user()
storage_client = storage.Client()
bucket = storage_client.bucket("processed_data-iyed")

def download_from_gcs(gcs_path, local_path):
    blob = bucket.blob(gcs_path)
    blob.download_to_filename(local_path)
    print(f"Downloaded {gcs_path} → {local_path}")

# Example:
# download_from_gcs("preprocessed/Apple.mat", "/content/Apple.mat")
# download_from_gcs("preprocessed/SpectroFood_dataset.csv", "/content/SpectroFood_dataset.csv")
"""


In [ ]:
# ============================================================
# CELL 3: LOAD APPLE DATA
# ============================================================

print("\n[1/5] Loading Apple.mat and SpectroFood_dataset.csv ...")

# ---- Load Apple.mat ----
mat_data = loadmat(config.APPLE_MAT_PATH)

apples = {}
for i in range(1, config.NUM_APPLES + 1):
    key = f"A{i}"
    if key not in mat_data:
        raise KeyError(f"Missing {key} in Apple.mat")

    apples[i - 1] = mat_data[key].astype(np.float32)

print(f"✓ Loaded {len(apples)} apples from Apple.mat")

# ---- Load CSV ----
csv_data = pd.read_csv(config.CSV_PATH)

apple_to_dm = {}
for _, row in csv_data.iterrows():
    apple_name = row["Apple"].strip()  # "A1"
    apple_id = int(apple_name[1:]) - 1
    apple_to_dm[apple_id] = float(row["Dry matter"])

dm_values = np.array([apple_to_dm[i] for i in range(config.NUM_APPLES)], dtype=np.float32)
print(f"✓ Loaded dry matter values. Range = {dm_values.min():.2f} to {dm_values.max():.2f}")



[1/10] Setting up GCS and downloading data...
  ✓ Downloaded X_spatial_train.h5 (29807.8 MB)
  ✓ Downloaded X_spatial_test.h5 (7809.9 MB)
  ✓ Downloaded X_spectral_train.npy (246.1 MB)
  ✓ Downloaded X_spectral_test.npy (64.0 MB)
  ✓ Downloaded Y_train.npy (5.2 MB)
  ✓ Downloaded Y_test.npy (1.4 MB)


In [ ]:
# ============================================================
# CELL 4: APPLE-LEVEL SPECTRAL FEATURES
# ============================================================

print("\n[2/5] Computing mean spectral signature for each apple...")

spectral_features = []

for apple_id, cube in apples.items():
    H, W, B = cube.shape
    pixels = cube.reshape(-1, B)
    mean_vec = pixels.mean(axis=0)
    spectral_features.append(mean_vec)

X_all = np.stack(spectral_features, axis=0)
y_dm = dm_values

print(f"✓ X_all shape = {X_all.shape}  # (240, 141)")



[2/10] Inspecting downloaded data...
Train spatial: (457463, 11, 11, 141)
Test spatial: (119016, 11, 11, 141)
Train spectral: (457463, 141)
Test spectral: (119016, 141)
Train labels: (457463, 3)
Test labels: (119016, 3)
Train class distribution: [143851 159321 154291]
Test class distribution: [23023 57210 38783]


In [ ]:
# ============================================================
# CELL 5: KMEANS → 3 MATURITY CLASSES
# ============================================================

print("\n[3/5] Running KMeans on dry matter values...")

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(y_dm.reshape(-1, 1))

cluster_means = [y_dm[clusters == c].mean() for c in range(3)]
order = np.argsort(cluster_means)

# Map clusters → class labels
labels = np.zeros_like(clusters)
for new_label, old_cluster in enumerate(order):
    labels[clusters == old_cluster] = new_label

print("✓ Class mapping (sorted by ripeness):")
for new_label, old_cluster in enumerate(order):
    print(f"  Class {new_label} ← cluster {old_cluster}, mean DM = {cluster_means[old_cluster]:.3f}")

print("Class counts:", np.bincount(labels))



[3/10] Creating custom data generators...
  ✓ Train generator: 7148 batches
  ✓ Val generator: 1860 batches


In [ ]:
# ============================================================
# CELL 6: SPLIT + NORMALIZE
# ============================================================

print("\n[4/5] Train/test split + scaling...")

idx = np.arange(config.NUM_APPLES)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, labels,
    test_size=config.TEST_SIZE,
    stratify=labels,
    random_state=config.RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_cat = keras.utils.to_categorical(y_train, 3)
y_test_cat = keras.utils.to_categorical(y_test, 3)

print("✓ X_train:", X_train_scaled.shape)
print("✓ X_test:", X_test_scaled.shape)



[4/10] Building hybrid CNN + Transformer model...


Model: "hybrid_cnn_transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ spatial_input       │ (None, 11, 11,    │          0 │ -                 │
│ (InputLayer)        │ 141)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spectral_input      │ (None, 141)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_cnn_branch  │ (None, 256)       │    495,872 │ spatial_input[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spectral_transform… │ (1, 256)          │  1,832,960 │ spectral_input[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_fusion    │ (1, 256)          │    263,680 │ spatial_cnn_bran… │
│ (AttentionFusion)   │                   │            │ spectral_transfo… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier_dense_0  │ (1, 512)          │    131,584 │ attention_fusion… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier_dropout… │ (1, 512)          │          0 │ classifier_dense… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier_dense_1  │ (1, 256)          │    131,328 │ classifier_dropo… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier_dropout… │ (1, 256)          │          0 │ classifier_dense… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier_dense_2  │ (1, 128)          │     32,896 │ classifier_dropo… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier_dropout… │ (1, 128)          │          0 │ classifier_dense… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (1, 3)            │        387 │ classifier_dropo… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,888,707 (11.02 MB)

 Trainable params: 2,887,747 (11.02 MB)

 Non-trainable params: 960 (3.75 KB)

  ✓ Model created with 2,888,707 parameters


In [ ]:
# ============================================================
# CELL 7: TRAIN SIMPLE MLP
# ============================================================

print("\n[5/5] Training spectral-only MLP...")

model = keras.Sequential([
    layers.Input(shape=(config.NUM_BANDS,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(config.LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint_path = os.path.join(config.CHECKPOINT_DIR, "best_model.h5")

cb = [
    callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor="val_accuracy", mode="max"),
    callbacks.EarlyStopping(monitor="val_accuracy", patience=20, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10),
]

history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_test_scaled, y_test_cat),
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    callbacks=cb,
    verbose=1
)


In [ ]:
# ============================================================
# CELL 8: EVALUATION
# ============================================================

print("\n📊 Final Evaluation")

loss, acc = model.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy = {acc:.4f}")

probs = model.predict(X_test_scaled)
preds = np.argmax(probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))
